In [1]:
import json
import concrete_ml_extensions
import concrete
import json
import concrete_ml_extensions as fhext
import json
from pathlib import Path 
from concrete import fhe 
import pickle as pkl 

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from concrete.ml.sklearn import DecisionTreeClassifier as ConcreteDecisionTreeClassifier
from sklearn.model_selection import train_test_split
from concrete.ml.common.utils import CiphertextFormat

assert concrete.ml.__version__ == "1.8.0"
assert concrete_ml_extensions.__version__ == "0.1.6"

x, y = make_classification(n_samples=100, class_sep=2, n_features=30, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)

model = ConcreteDecisionTreeClassifier(n_bits=8)
model.fit(X_train, y_train)

model.compile(X_train[:10, :], ciphertext_format=CiphertextFormat.TFHE_RS)

crypto_params = json.loads(fhext.get_crypto_params_radix())

print(crypto_params)

sk, tfhers_pk, pk, lwe_sk = model.keygen_tfhers()

q_input = model.quantize_input(X_test[[0]])

q_input_enc = model.encrypt_tfhers(q_input, tfhers_sk=sk)

q_y_enc = model.run_tfhers(q_input_enc, tfhers_pk=tfhers_pk)

q_y = model.decrypt_tfhers(q_y_enc, tfhers_sk=sk)

y_pred = model.post_processing(model.dequantize_output(q_y))

assert y_pred.argmax() == y_test[0]

No CUDA runtime is found, using CUDA_HOME='/usr/local/cuda'


----> {'lwe_dimension': 834, 'glwe_dimension': 1, 'polynomial_size': 2048, 'lwe_noise_distribution': {'Gaussian': {'std': 3.5539902359442825e-06, 'mean': 0.0}}, 'glwe_noise_distribution': {'Gaussian': {'std': 2.845267479601915e-15, 'mean': 0.0}}, 'pbs_base_log': 23, 'pbs_level': 1, 'ks_base_log': 3, 'ks_level': 5, 'message_modulus': 4, 'carry_modulus': 4, 'max_noise_level': 5, 'log2_p_fail': -64.074, 'ciphertext_modulus': {'modulus': 0, 'scalar_bits': 64}, 'encryption_key_choice': 'Big', 'modulus_switch_noise_reduction_params': None}
{'lwe_dimension': 834, 'glwe_dimension': 1, 'polynomial_size': 2048, 'lwe_noise_distribution': {'Gaussian': {'std': 3.5539902359442825e-06, 'mean': 0.0}}, 'glwe_noise_distribution': {'Gaussian': {'std': 2.845267479601915e-15, 'mean': 0.0}}, 'pbs_base_log': 23, 'pbs_level': 1, 'ks_base_log': 3, 'ks_level': 5, 'message_modulus': 4, 'carry_modulus': 4, 'max_noise_level': 5, 'log2_p_fail': -64.074, 'ciphertext_modulus': {'modulus': 0, 'scalar_bits': 64}, 'encr

RuntimeError: Tried to transform transport value with incompatible type info.
Expected: (lweCiphertext = (abstractShape = (dimensions = [1, 30, 4]), concreteShape = (dimensions = [1, 30, 4, 2049]), integerPrecision = 64, encryption = (keyId = 0, variance = 8.4422531129329586e-31, lweDimension = 2048, modulus = (modulus = (native = ()))), compression = none, encoding = (integer = (width = 4, isSigned = false, mode = (native = ())))))
Actual: (lweCiphertext = (abstractShape = (dimensions = [1, 30, 4]), concreteShape = (dimensions = [1, 30, 4, 2049]), integerPrecision = 64, encryption = (keyId = 0, variance = 8.0955470304802333e-30, lweDimension = 2048, modulus = (modulus = (native = ()))), compression = none, encoding = (integer = (width = 4, isSigned = false, mode = (native = ())))))

In [ ]:
!pip list | grep concrete

concrete-ml                   1.8.0                /home/celia/Desktop/Zama/concrete-internal
concrete-ml-extensions        0.1.6                /home/celia/Desktop/Zama/concrete-internal/use_case_examples/tfhers_demo/concrete-ml-extensions
concrete-python               2.9.0rc2.dev20250305
tfhers_demo                   0.1.0                /home/celia/Desktop/Zama/concrete-internal/use_case_examples/tfhers_demo


In [ ]:
# with open("secreteKey.bin", "wb") as f:
#     f.write(sk)

# with open("encrypted_input.bin", "wb") as f:
#     f.write(q_y_enc)
    
# with open("lweServerKey.bin", "wb") as f:
#     f.write(lwe_sk)
    
with open("evalkeys.bin", "wb") as f:
    f.write(tfhers_pk.serialize())
    
with open("input.bin", "wb") as f:
    f.write(q_input_enc[0].serialize())
    
path_circuit_server = Path("server").joinpath("server.zip")
model.fhe_circuit.server.save(path_circuit_server, via_mlir=True)

# with open("mlir_circuit.pkl", "wb") as f:
#     pkl.dump(model.fhe_circuit.mlir, f)

In [2]:
# test concrete code
# 

server = fhe.Server.load("server/server.zip")

with open("input.bin", "rb") as f:
    b = f.read()
value = fhe.Value.deserialize(b)

with open("evalkeys.bin", "rb") as f:
    b = f.read()
evalKeys = fhe.EvaluationKeys.deserialize(b)

server.run(value, evaluation_keys=evalKeys)

RuntimeError: Tried to transform transport value with incompatible type info.
Expected: (lweCiphertext = (abstractShape = (dimensions = [1, 30, 4]), concreteShape = (dimensions = [1, 30, 4, 2049]), integerPrecision = 64, encryption = (keyId = 0, variance = 8.4422531129329586e-31, lweDimension = 2048, modulus = (modulus = (native = ()))), compression = none, encoding = (integer = (width = 4, isSigned = false, mode = (native = ())))))
Actual: (lweCiphertext = (abstractShape = (dimensions = [1, 30, 4]), concreteShape = (dimensions = [1, 30, 4, 2049]), integerPrecision = 64, encryption = (keyId = 0, variance = 8.0955470304802333e-30, lweDimension = 2048, modulus = (modulus = (native = ()))), compression = none, encoding = (integer = (width = 4, isSigned = false, mode = (native = ())))))